# AWS role 구현

- AWS에서 role을 구현하려면 
  - 신뢰정책과 권한정책이 필요하다.
  - 권한정책은 3가지 방식으로 구현할 수 있다.
    - 인라인: role 내부에 내장되어 자체 ARN 없음, role 삭제 시 함께 소멸, 재사용 불가
    - 고객관리형: 자체 ARN을 가지는 독립 객체. 사용자가 정의, 변경, 삭제 가능, 재사용 가능
    - AWS관리형: AWS가 제공하는 독립 객체. attach/detach만 가능하고 내용 수정 및 삭제 불가, 재사용 가능
  - 신뢰정책은 인라인 방식으로만 구현할 수 있다.



- AWS에서 role을 구현할 때 사용되는 리소스는 다음과 같다.
  - `aws_iam_policy_document`: 신뢰정책이나 권한정책 등 정책의 내용 정의
  - `aws_iam_role`: role 자체의 정의 + 신뢰정책과의 연결
  - `aws_iam_role_policy`: 인라인 권한정책 정의 + role과의 연결 
  - `aws_iam_policy`: 고객관리형 권한정책의 정의 
  - `aws_iam_role_policy_attachment`: 고객관리형 권한정책이나 AWS관리형 권한정책을 role과 연결

- 구현 단계
  - (1단계) `aws_iam_policy_document` 리소스로 신뢰정책이나 권한정책 등의 내용을 정의한다.
    - `jsonencode` 방식으로 정책 내용을 인수에 직접 넣을 때는 생략할 수 있다.
  - (2단계) `aws_iam_role` 리소스로 role 자체를 정의하고 신뢰정책을 연결한다.
  - (3단계) `aws_iam_role_policy` 또는 `aws_iam_policy` 리소스로 권한정책을 정의한다.
    - AWS 관리형 권한정책만 사용하는 경우에는 이 단계가 필요없다.
  - (4단계) `aws_iam_role_policy_attachment` 로 권한정책을 role과 연결한다.
    - 만약 권한정책을 여러개 연결하는 경우 (3단계)~(4단계)를 반복한다.

## `aws_iam_policy_document` 정보

- 신뢰대상 정책이나 권한 정책 등 정책의 내용을 정의하는 정보
- 리소스가 아니라 정보이므로 항상 `data` 로 정의함
- 인수
  - statement: 정책의 내용을 정의
    - actions: 어떤 정책인지 정의. `sts:`로 시작하면 신뢰 정책
    - principals: 신뢰 정책의 경우에 누구를 신뢰할지 주체를 정의
    - resources: 권한 정책의 경우에 권한의 대상을 정의

- 예시: 신뢰 정책 내용의 정의
  ```yaml
  data "aws_iam_policy_document" "event_stream_bucket_role_assume_role_policy" {
    statement {
      actions = ["sts:AssumeRole"]

      principals {
        type        = "Service"
        identifiers = ["firehose.amazonaws.com"]
      }
    }
  }
  ```

- 예시: 권한 정책 내용의 정의
  ```yaml
  data "aws_iam_policy_document" "example" {
    statement {
      actions = [
        "s3:*",
      ]

      resources = [
        "arn:aws:s3:::${var.s3_bucket_name}/home/&{aws:username}",
        "arn:aws:s3:::${var.s3_bucket_name}/home/&{aws:username}/*",
      ]
    }
  }
  ```

- `aws_iam_policy_document` 정보는 `aws_iam_role`,  `aws_iam_role_policy`, `aws_iam_policy` 리소스에서 인수로 사용되는 값인데
- `aws_iam_policy_document` 정보를 별도로 정의하지 않고 `jsonencode` 방식으로 인수에 직접 넣을 수도 있다.

## `aws_iam_role` 리소스

- role 자체와 신뢰정책을 정의하는 리소스
- 인수
  - `assume_role_policy`
    - 신뢰정책 설정
    - 미리 만들어진 aws_iam_policy_document 의 json 속성을 연결하거나 jsonencode로 직접 정의

```yaml
resource "aws_iam_role" "example" {
  name                = "yak_role"
  assume_role_policy  = data.aws_iam_policy_document.instance_assume_role_policy.json # (not shown)
}
```

## `aws_iam_role_policy` 리소스

- 인라인 권한정책을 만들면서 role과 연결까지 한다.
- 인수
  - policy: 권한정책의 내용
  - role: 연결할 role

```yaml
resource "aws_iam_role_policy" "stop_nodes_ec2" {
  name   = "sch-nodes-stop-ec2"
  role   = aws_iam_role.stop_nodes_lambda.id
  policy = data.aws_iam_policy_document.stop_nodes_ec2.json
}
```

## `aws_iam_policy` 리소스

- 고객관리형 권한정책의 정의 

## `aws_iam_role_policy_attachment` 리소스

- 고객관리형 권한정책이나 AWS관리형 권한정책을 role과 연결